# Review Sentiment Loader

- Select `METHOD = "rule"` or `METHOD = "model"` to compute review-level sentiment and write it to `review_sentiment`.
- Set `OVERWRITE = True` to recompute all reviews, or `False` to only insert rows for reviews without sentiment.

In [4]:
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

DB_PATH = Path("../data/app.sqlite")
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

In [16]:
RULE_VERSION = "v1"

# METHOD = "rule"
METHOD = "model"

OVERWRITE = False

In [6]:
import re

LEXICON = {
    "good": 1.0,
    "great": 1.5,
    "amazing": 2.0,
    "excellent": 2.0,
    "love": 1.5,
    "like": 1.0,
    "bad": -1.0,
    "terrible": -2.0,
    "awful": -2.0,
    "hate": -1.5,
    "dislike": -1.0,
    "poor": -1.0,
    "worst": -2.0,
    "좋다": 1.0,
    "최고": 2.0,
    "훌륭": 1.5,
    "맛있": 1.5,
    "추천": 1.0,
    "만족": 1.0,
    "친절": 1.0,
    "별로": -1.0,
    "나쁘": -1.0,
    "최악": -2.0,
    "불친절": -1.5,
    "실망": -1.0,
}


def _tokenize(text: str) -> list[str]:
    return re.findall(r"[A-Za-z']+|[가-힣]+", (text or "").lower())


def compute_rule_sentiment(text: str) -> tuple[float, float, str, str]:
    tokens = _tokenize(text)
    if not tokens:
        return 0.0, 0.0, "rule", RULE_VERSION
    score = 0.0
    matched = 0
    for token in tokens:
        weight = LEXICON.get(token)
        if weight is None:
            continue
        matched += 1
        score += weight
    confidence = matched / len(tokens)
    return score, confidence, "rule", RULE_VERSION


def load_model_pipeline():
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        use_safetensors=True,
    )
    return pipeline(
        "sentiment-analysis",
        model=model,
        tokenizer=tokenizer,
        truncation=True,
        max_length=256,
        batch_size=16,
    )


def compute_model_sentiment(text: str, pipe) -> tuple[float, float, str, str]:
    result = pipe(text, truncation=True)[0]
    label = str(result.get("label", "")).lower()
    score = float(result.get("score", 0.0))
    if "negative" in label:
        polarity = -score
    elif "positive" in label:
        polarity = score
    else:
        polarity = 0.0
    return polarity, score, "model", MODEL_NAME

In [7]:
conn = sqlite3.connect(DB_PATH)

In [14]:
pd.read_sql(
        "SELECT * FROM review_sentiment",
        conn,
    )

,review_id,sentiment_raw,confidence,method,version,created_at


In [13]:
conn.executescript(
#        """ DELETE FROM sqlite_sequence WHERE name = 'review_sentiment'; """
#        """ DELETE FROM review_sentiment; """
)
conn.commit()

In [17]:
if not DB_PATH.exists():
    raise FileNotFoundError(f"DB not found at {DB_PATH.resolve()}")
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")
def ensure_review_sentiment(conn):
    conn.executescript(
        """
        CREATE TABLE IF NOT EXISTS review_sentiment (
          review_id INTEGER NOT NULL,
          sentiment_raw REAL NOT NULL,
          confidence REAL NOT NULL,
          method TEXT NOT NULL,
          version TEXT,
          created_at TEXT,
          PRIMARY KEY (review_id, method, version),
          FOREIGN KEY (review_id) REFERENCES review(id)
        );
        """
    )
ensure_review_sentiment(conn)
reviews = pd.read_sql(
    "SELECT id AS review_id, content FROM review",
    conn,
)


In [18]:
print(f"Reviews queued for sentiment: {len(reviews)}")
if len(reviews) > 0:
    display(reviews.head(10))

Reviews queued for sentiment: 43


,review_id,content
0,1,BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마...
1,2,처가집 양념통닭에서 마라치킨 주문. 그냥 달달한 양념 치킨에 마라향 살짝 추가한 정...
2,3,배달앱에서 멕시카나 치킨 할인한다기에 주문하려했더니 미금점 영업을 하지 않는다. 이...
3,4,비비큐에서 맵소디 배달 주문. 양념 맛은 기억나지 않지만 소스 안쪽에서도 바삭한 식...
4,5,노랑통닭은 순살도 가격이 더 비싸지 않아 좋다. 튀김 부스러기는 과자같이 바삭하니 ...
5,6,푸라닭에서 유명 쉐프와 협업했다는 깐풍치킨 주문. 소스가 볶은 느낌이 아니라 흥건하...
6,7,교촌치킨에서 간장 레드 반반 주문. 간장만 먹으면 심심하고 레드만 먹으면 자극적이면...
7,8,땡겨요에서 할인을 크게 해서 한동안 집앞 자담치킨 참 많이도 포장해 먹었다. 왓더핫...
8,9,굽네치킨에서 유일하게 좋아하는 고추바사삭. 근데 살이 너무 없는것 같다
9,10,성남 현대시장 명성왕족발에서 족발 대자 포장. 싸고 맛 좋고 특히 온누리상품권을 쓸...


In [19]:
if reviews.empty:
    raise ValueError("review is empty; nothing to compute.")
if not OVERWRITE:
    existing = pd.read_sql(
        "SELECT DISTINCT review_id FROM review_sentiment",
        conn,
    )
    if not existing.empty:
        reviews = reviews[~reviews["review_id"].isin(existing["review_id"])]
if reviews.empty:
    conn.close()
    print("No reviews to compute sentiment for.")
else:
    print(f"Reviews to compute sentiment: {len(reviews)}")
    display(reviews.head(10))

Reviews to compute sentiment: 43


,review_id,content
0,1,BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마...
1,2,처가집 양념통닭에서 마라치킨 주문. 그냥 달달한 양념 치킨에 마라향 살짝 추가한 정...
2,3,배달앱에서 멕시카나 치킨 할인한다기에 주문하려했더니 미금점 영업을 하지 않는다. 이...
3,4,비비큐에서 맵소디 배달 주문. 양념 맛은 기억나지 않지만 소스 안쪽에서도 바삭한 식...
4,5,노랑통닭은 순살도 가격이 더 비싸지 않아 좋다. 튀김 부스러기는 과자같이 바삭하니 ...
5,6,푸라닭에서 유명 쉐프와 협업했다는 깐풍치킨 주문. 소스가 볶은 느낌이 아니라 흥건하...
6,7,교촌치킨에서 간장 레드 반반 주문. 간장만 먹으면 심심하고 레드만 먹으면 자극적이면...
7,8,땡겨요에서 할인을 크게 해서 한동안 집앞 자담치킨 참 많이도 포장해 먹었다. 왓더핫...
8,9,굽네치킨에서 유일하게 좋아하는 고추바사삭. 근데 살이 너무 없는것 같다
9,10,성남 현대시장 명성왕족발에서 족발 대자 포장. 싸고 맛 좋고 특히 온누리상품권을 쓸...


In [20]:
if not reviews.empty:
    if METHOD == "model":
        pipe = load_model_pipeline()
        sentiment = reviews["content"].apply(lambda text: compute_model_sentiment(text, pipe))
    elif METHOD == "rule":
        sentiment = reviews["content"].apply(compute_rule_sentiment)
    else:
        raise ValueError(f"Unknown METHOD: {METHOD}")
    reviews["sentiment_raw"] = sentiment.map(lambda item: item[0])
    reviews["confidence"] = sentiment.map(lambda item: item[1])
    reviews["method"] = sentiment.map(lambda item: item[2])
    reviews["version"] = sentiment.map(lambda item: item[3])
    reviews["created_at"] = datetime.now(timezone.utc).isoformat()
    payload = list(
        reviews[
            [
                "review_id",
                "sentiment_raw",
                "confidence",
                "method",
                "version",
                "created_at",
            ]
        ].itertuples(index=False, name=None)
    )
    conn.executemany(
        """
        INSERT OR REPLACE INTO review_sentiment
        (review_id, sentiment_raw, confidence, method, version, created_at)
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        payload,
    )
    conn.commit()
    conn.close()
    print(f"Inserted/updated {len(payload)} sentiment rows using METHOD={METHOD}.")

Device set to use cpu


Inserted/updated 43 sentiment rows using METHOD=model.


In [21]:
conn = sqlite3.connect(DB_PATH)

pd.read_sql(
        "SELECT * FROM review_sentiment",
        conn,
    )

,review_id,sentiment_raw,confidence,method,version,created_at
0,1,0.530003,0.530003,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00
1,2,0.000000,0.379968,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00
2,3,-0.635604,0.635604,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00
3,4,0.000000,0.572736,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00
4,5,0.694385,0.694385,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00
5,6,0.643638,0.643638,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00
6,7,0.634538,0.634538,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00
7,8,0.000000,0.360365,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00
8,9,-0.496433,0.496433,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00
9,10,0.814621,0.814621,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00
